# DS102 — bản cải thiện v2 (CPU)

Mục tiêu: F1-macro của 4 nhãn spam. Dùng TF-IDF từ/ký tự, đặc trưng review, mức trùng với tên/mô tả sản phẩm, rating, category, num_sold và num_review. Không cần GPU, pretrained model hoặc dữ liệu ngoài.

**Cách dùng:** Import notebook này → Add Input dữ liệu cuộc thi → Run All. Các thư viện cần thiết là numpy, pandas, scipy và scikit-learn (1.5–1.7). Notebook tự chứa code, không cần upload các file `.py`.

Cấu hình mô hình đã được chọn bằng thử nghiệm cross-validation trên train. Notebook chạy lại 3-fold StratifiedGroupKFold cho cấu hình này, kiểm tra calibration bằng các dự đoán OOF, fit toàn bộ train, đánh giá dev rồi tạo `submission_v2.csv`. Comment giống nhau sau chuẩn hóa nằm cùng fold. Mọi vocabulary/IDF/scaler/trọng số mô hình chỉ học từ phần train tương ứng. Không gộp dev vào train.

CV được dùng chọn mô hình nên là ước lượng phục vụ phát triển, không phải một test độc lập. Dev đã được dùng xem lỗi bản cũ; điểm dev không bảo đảm điểm leaderboard. Mỗi lần chạy giữ seed=102. Chưa xác minh được Rules về thư viện của môn học.


## 1. Đọc dữ liệu, chuẩn hóa và tính metric

In [ ]:
"""CPU baseline cho DS102: TF-IDF + Linear SVM, chọn theo F1-macro.

Chỉ dùng comment và nhãn train. Dev được đánh giá sau khi chọn tham số
bằng cross-validation trên train; không gộp dev vào train.
"""

import argparse
import html
import json
import platform
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC


SEED = 102
LABELS = [0, 1, 2, 3]
FILES = {name: f"ds102_{name}.csv" for name in ["train", "dev", "test", "samplesubmission"]}


def normalize_text(value):
    """Giữ dấu, emoji, dấu câu và ký tự lặp vì có thể giúp nhận diện spam."""
    text = html.unescape(unicodedata.normalize("NFC", str(value)))
    text = text.replace("\\r\\n", "\n").replace("\\n", "\n").replace("\\r", "\n")
    return re.sub(r"\s+", " ", text).strip().lower()


class ReviewStats(BaseEstimator, TransformerMixin):
    """Đặc trưng độ dài, lặp ký tự và tỷ lệ ký tự; không có từ khóa nhãn."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        rows = []
        for text in X:
            chars = len(text)
            words = text.split()
            nwords = len(words)
            denom = max(chars, 1)
            word_denom = max(nwords, 1)
            letters = [c for c in text if c.isalpha()]
            longest_run = max((len(m.group()) for m in re.finditer(r"(.)\1*", text)), default=0)
            rows.append([
                np.log1p(chars), np.log1p(nwords), chars / word_denom,
                len(set(words)) / word_denom, len(set(text)) / denom,
                longest_run / denom, np.log1p(longest_run),
                sum(c.isdigit() for c in text) / denom,
                len(letters) / denom,
                sum(not c.isalnum() and not c.isspace() for c in text) / denom,
                sum(ord(c) > 0xFFFF for c in text) / denom,
                text.count("!") / denom, text.count("?") / denom,
                float(bool(re.search(r"https?://|www\.", text))),
            ])
        return sparse.csr_matrix(np.asarray(rows, dtype=np.float32))


def make_features():
    return FeatureUnion([
        ("word", TfidfVectorizer(
            ngram_range=(1, 2), min_df=2, max_df=1.0, max_features=90000,
            sublinear_tf=True, lowercase=False, dtype=np.float32,
            token_pattern=r"(?u)\b\w+\b",
        )),
        ("char", TfidfVectorizer(
            analyzer="char", ngram_range=(2, 5), min_df=3, max_features=160000,
            sublinear_tf=True, lowercase=False, dtype=np.float32,
        )),
        ("stats", Pipeline([
            ("extract", ReviewStats()), ("scale", StandardScaler(with_mean=False)),
        ])),
    ], transformer_weights={"word": 1.0, "char": 1.0, "stats": 0.1})


def make_model(config, y):
    weight = config["weight"]
    if weight == "sqrt_balanced":
        counts = pd.Series(y).value_counts()
        weight = {int(k): float(np.sqrt(len(y) / (len(counts) * v))) for k, v in counts.items()}
    elif weight == "none":
        weight = None
    return LinearSVC(C=config["C"], class_weight=weight, dual=True,
                     max_iter=10000, tol=1e-4, random_state=SEED)


def find_data_dir(supplied=None):
    if supplied:
        path = Path(supplied).expanduser().resolve()
        if all((path / f).is_file() for f in FILES.values()):
            return path
        raise FileNotFoundError(f"Thiếu một trong 4 file CSV tại {path}")
    candidates = [Path.cwd(), Path.cwd() / "data", Path.cwd() / "dataset"]
    kaggle = Path("/kaggle/input")
    if kaggle.exists():
        candidates += sorted({p.parent for p in kaggle.rglob(FILES["train"])})
    matches = [p for p in candidates if all((p / f).is_file() for f in FILES.values())]
    if len(matches) == 1:
        return matches[0].resolve()
    raise FileNotFoundError("Hãy truyền --data-dir đến thư mục chứa đúng 4 CSV; trên Kaggle dùng Add Input trước.")


def load_data(data_dir):
    data = {name: pd.read_csv(data_dir / file, keep_default_na=False) for name, file in FILES.items()}
    for name in ["train", "dev", "test"]:
        required = {"comment", "spam_label"} if name != "test" else {"comment", "id"}
        if not required.issubset(data[name].columns):
            raise ValueError(f"{name}: thiếu cột {required - set(data[name].columns)}")
        if data[name].empty:
            raise ValueError(f"{name} rỗng")
    for name in ["train", "dev"]:
        values = pd.to_numeric(data[name]["spam_label"], errors="raise")
        if not values.isin(LABELS).all():
            raise ValueError(f"{name}: nhãn phải thuộc {LABELS}")
        data[name]["spam_label"] = values.astype(int)
    if set(data["train"]["spam_label"]) != set(LABELS):
        raise ValueError("Train phải có đủ 4 nhãn")
    sample = data["samplesubmission"]
    if list(sample.columns) != ["ID", "spam_label"]:
        raise ValueError("Sample phải có hai cột ID,spam_label")
    test_ids, sample_ids = data["test"]["id"].astype(str), sample["ID"].astype(str)
    if not test_ids.is_unique or not sample_ids.is_unique or set(test_ids) != set(sample_ids):
        raise ValueError("ID trong test và sample phải duy nhất và trùng tập hợp")
    return data


def metrics(y, pred):
    return {
        "macro_f1": float(f1_score(y, pred, labels=LABELS, average="macro", zero_division=0)),
        "accuracy": float(accuracy_score(y, pred)),
        "classification_report": classification_report(y, pred, labels=LABELS, output_dict=True, zero_division=0),
        "confusion_matrix": confusion_matrix(y, pred, labels=LABELS).tolist(),
    }



## 2. Đặc trưng văn bản và quan hệ review/sản phẩm

In [ ]:
"""Đặc trưng v2: văn bản review và mức trùng với thông tin sản phẩm."""
import re
from collections import Counter
import numpy as np
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


def prepare(frame):
    frame = frame.copy()
    for col in ['comment', 'product_name', 'description', 'category', 'categories']:
        frame[col] = frame[col].fillna('').astype(str).map(normalize_text)
    return frame


def tokens(text):
    return re.findall(r'(?u)\b\w+\b', text)


def ngrams(words, n):
    return set(zip(*(words[i:] for i in range(n))))


def dense_features(df):
    basic = ReviewStats().transform(df.comment).toarray()
    extra = []
    for row in df.itertuples(index=False):
        text = row.comment
        words = tokens(text)
        wl = [len(w) for w in words] or [0]
        wc = Counter(words)
        n = max(len(words), 1)
        feats = [np.log1p(max(wl)), np.mean(wl), np.std(wl),
                 sum(len(w) > 12 for w in words) / n,
                 max(wc.values(), default=0) / n,
                 sum(v * (v - 1) for v in wc.values()) / max(n * (n - 1), 1),
                 text.count('.') / max(len(text), 1),
                 len(re.findall(r'(.)\1{3,}', text)) / n,
                 np.log1p(len(text.split())),
                 float(row.rating) / 5,
                 np.log1p(max(float(row.num_sold), 0)),
                 np.log1p(max(float(row.num_review), 0))]
        for context in [row.product_name, row.description]:
            context_words = tokens(context)
            for size in [1, 2, 3, 4]:
                a, b = ngrams(words, size), ngrams(context_words, size)
                overlap = len(a & b)
                ratio = overlap / max(len(a), 1)
                feats += [ratio, ratio ** 2, ratio * np.log1p(len(words)), np.log1p(overlap)]
            feats += [float(len(text) > 10 and text in context), np.log1p(len(context))]
        extra.append(feats)
    return np.hstack([basic, np.asarray(extra, dtype=np.float32)])


class FeaturesV2:
    def __init__(self):
        self.word = TfidfVectorizer(ngram_range=(1, 3), min_df=2, max_features=140000,
                                   sublinear_tf=True, lowercase=False, dtype=np.float32,
                                   token_pattern=r'(?u)\b\w+\b')
        self.char = TfidfVectorizer(analyzer='char', ngram_range=(2, 5), min_df=3,
                                   max_features=180000, sublinear_tf=True,
                                   lowercase=False, dtype=np.float32)
        self.product = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=20000,
                                      sublinear_tf=True, lowercase=False, dtype=np.float32)
        self.category = OneHotEncoder(handle_unknown='ignore', dtype=np.float32)
        self.scale = StandardScaler()

    def fit_transform(self, df):
        text = sparse.hstack([self.word.fit_transform(df.comment), self.char.fit_transform(df.comment)], format='csr')
        meta = sparse.hstack([self.product.fit_transform(df.product_name) * .2,
                             self.category.fit_transform(df[['category', 'rating']].astype(str)) * .2,
                             sparse.csr_matrix(self.scale.fit_transform(dense_features(df))) * .15], format='csr')
        return text, meta

    def transform(self, df):
        text = sparse.hstack([self.word.transform(df.comment), self.char.transform(df.comment)], format='csr')
        meta = sparse.hstack([self.product.transform(df.product_name) * .2,
                             self.category.transform(df[['category', 'rating']].astype(str)) * .2,
                             sparse.csr_matrix(self.scale.transform(dense_features(df))) * .15], format='csr')
        return text, meta


def nb_ratios(X, y):
    """Log-count ratio one-vs-rest, fit chỉ từ phần train của fold."""
    ratios = []
    for c in range(4):
        pos = np.asarray(X[y == c].sum(axis=0)).ravel() + 1
        neg = np.asarray(X[y != c].sum(axis=0)).ravel() + 1
        ratios.append(np.log((pos / pos.sum()) / (neg / neg.sum())))
    return ratios

## 3. Điều chỉnh ngưỡng trên OOF train

In [ ]:
"""Chọn mô hình/ensemble và bias bằng OOF train, không xem nhãn dev."""
import json
from pathlib import Path
import numpy as np


def macro(y, pred):
    cm = np.bincount(4 * y + pred, minlength=16).reshape(4, 4)
    den = cm.sum(axis=0) + cm.sum(axis=1)
    return float(np.mean(np.divide(2 * np.diag(cm), den, out=np.zeros(4), where=den > 0)))


def calibrate(scores, y):
    # 3 offsets tương đối so với nhãn 0; grid nhỏ để hạn chế overfit.
    bias = np.zeros(4)
    for _ in range(3):
        previous = bias.copy()
        for cls in [1, 2, 3]:
            choices = []
            for value in np.linspace(-.4, .4, 17):
                proposal = bias.copy()
                proposal[cls] = value
                objective = macro(y, (scores + proposal).argmax(axis=1)) - .005 * np.square(proposal).sum()
                choices.append((objective, -abs(value), float(value)))
            bias[cls] = max(choices)[2]
        if np.allclose(previous, bias):
            break
    return bias



## 4. Pipeline huấn luyện và kiểm tra submission

In [ ]:
"""DS102 v2: đặc trưng quan hệ review/sản phẩm, SVM và calibration F1-macro."""
import argparse
import hashlib
import json
import platform
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from scipy import sparse
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.svm import LinearSVC


# Cấu hình được chốt từ cross-validation trên train, trước đánh giá dev v2.
RECIPE = [
    {'kind': 'svm', 'C': .5, 'weight': 'balanced', 'ensemble_weight': .75},
    {'kind': 'nb', 'C': .5, 'weight': 'balanced', 'ensemble_weight': .25},
]


def fit_predict(fit_df, predict_frames, recipe):
    """Không nhận nhãn của các tập dự đoán."""
    y = fit_df.spam_label.to_numpy()
    features = FeaturesV2()
    Xt, Mt = features.fit_transform(fit_df)
    transformed = [features.transform(df) for df in predict_frames]
    result = [np.zeros((len(df), 4)) for df in predict_frames]
    ratios = nb_ratios(Xt, y) if any(c['kind'] == 'nb' for c in recipe) else None
    for config in recipe:
        if config['kind'] == 'svm':
            model = make_model(config, y).fit(sparse.hstack([Xt, Mt], format='csr'), y)
            if np.max(model.n_iter_) >= model.max_iter:
                raise RuntimeError('SVM chưa hội tụ; tăng max_iter trước khi xuất kết quả.')
            for j, (Xv, Mv) in enumerate(transformed):
                result[j] += config['ensemble_weight'] * model.decision_function(sparse.hstack([Xv, Mv], format='csr'))
        elif config['kind'] == 'nb':
            for cls, r in enumerate(ratios):
                model = LinearSVC(C=config['C'], class_weight='balanced', dual=True,
                                  max_iter=30000, random_state=SEED)
                model.fit(sparse.hstack([Xt.multiply(r), Mt], format='csr'), (y == cls).astype(int))
                if np.max(model.n_iter_) >= model.max_iter:
                    raise RuntimeError('NB-SVM chưa hội tụ; tăng max_iter trước khi xuất kết quả.')
                for j, (Xv, Mv) in enumerate(transformed):
                    result[j][:, cls] += config['ensemble_weight'] * model.decision_function(
                        sparse.hstack([Xv.multiply(r), Mv], format='csr'))
        else:
            raise ValueError(f"Model không được hỗ trợ: {config['kind']}")
    return result


def main_v2(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--data-dir', default=None)
    parser.add_argument('--output-dir', default='output_v2')
    parser.add_argument('--folds', type=int, default=3)
    args = parser.parse_args(argv)
    if args.folds < 2:
        parser.error('--folds phải >= 2')
    data_dir = find_data_dir(args.data_dir)
    data = load_data(data_dir)
    expected = {'comment', 'product_name', 'description', 'category', 'categories', 'rating', 'num_sold', 'num_review'}
    for name in ['train', 'dev', 'test']:
        missing = expected - set(data[name].columns)
        if missing:
            raise ValueError(f'{name} thiếu cột {missing}')
        for col in ['rating', 'num_sold', 'num_review']:
            values = pd.to_numeric(data[name][col], errors='raise')
            if not np.isfinite(values).all() or (values < 0).any():
                raise ValueError(f'{name}.{col}: giá trị số không hợp lệ')
            data[name][col] = values
    train, dev, test = [prepare(data[n]) for n in ['train', 'dev', 'test']]
    output = Path(args.output_dir).expanduser().resolve()
    output.mkdir(parents=True, exist_ok=True)
    y = train.spam_label.to_numpy()
    oof = np.zeros((len(train), 4))
    fold_ids = np.full(len(train), -1)
    splitter = StratifiedGroupKFold(n_splits=args.folds, shuffle=True, random_state=SEED)
    for fold, (tr, va) in enumerate(splitter.split(train, y, groups=train.comment)):
        if set(y[tr]) != set(LABELS):
            raise ValueError('Fold không đủ 4 lớp; giảm số folds.')
        print(f'CV fold {fold+1}/{args.folds}', flush=True)
        oof[va] = fit_predict(train.iloc[tr], [train.iloc[va].drop(columns='spam_label')], RECIPE)[0]
        fold_ids[va] = fold
        print(f'  Raw F1-macro={macro(y[va], oof[va].argmax(axis=1)):.5f}', flush=True)
    cross_pred = np.zeros(len(train), dtype=int)
    for f in range(args.folds):
        fit = fold_ids != f
        bias_fold = calibrate(oof[fit], y[fit])
        cross_pred[~fit] = (oof[~fit] + bias_fold).argmax(axis=1)
    raw_score = macro(y, oof.argmax(axis=1))
    cross_score = macro(y, cross_pred)
    use_bias = cross_score > raw_score + .002
    bias = calibrate(oof, y) if use_bias else np.zeros(4)
    print(f'OOF raw={raw_score:.5f}; cross-calibrated={cross_score:.5f}; bias={bias.tolist()}', flush=True)
    print('Fit toàn bộ train và dự đoán dev/test...', flush=True)
    dev_scores, test_scores = fit_predict(train, [dev.drop(columns='spam_label'), test], RECIPE)
    dev_pred = (dev_scores + bias).argmax(axis=1)
    test_pred = (test_scores + bias).argmax(axis=1)
    dev_y = dev.spam_label.to_numpy()
    unique_dev = ~dev.comment.isin(set(train.comment)).to_numpy()
    dev_report = metrics(dev_y, dev_pred)
    print(f"Dev F1-macro={dev_report['macro_f1']:.5f}", flush=True)
    print(classification_report(dev_y, dev_pred, labels=LABELS, digits=4, zero_division=0), flush=True)
    by_id = pd.Series(test_pred, index=test.id.astype(str))
    submission = data['samplesubmission'].copy()
    submission['spam_label'] = submission.ID.astype(str).map(by_id)
    if submission.spam_label.isna().any() or not submission.spam_label.isin(LABELS).all():
        raise ValueError('Nhãn submission không hợp lệ')
    submission['spam_label'] = submission.spam_label.astype(int)
    path = output/'submission_v2.csv'
    submission.to_csv(path, index=False)
    saved = pd.read_csv(path)
    pd.testing.assert_frame_equal(saved, submission, check_dtype=False)
    if len(saved) != len(test) or not saved.ID.equals(data['samplesubmission'].ID):
        raise ValueError('Sai số dòng hoặc thứ tự ID')
    pd.DataFrame({'comment': data['dev'].comment, 'true_label': dev_y,
                  'predicted_label': dev_pred, 'comment_seen_in_train': ~unique_dev}).to_csv(output/'dev_predictions_v2.csv', index=False)
    np.savez_compressed(output/'scores_v2.npz', oof=oof, y=y, folds=fold_ids,
                        dev_scores=dev_scores, test_scores=test_scores, bias=bias)
    report = {
        'recipe': RECIPE, 'seed': SEED, 'folds': args.folds, 'python': platform.python_version(),
        'sklearn': sklearn.__version__, 'data_sha256': {name: hashlib.sha256((data_dir/file).read_bytes()).hexdigest()
            for name, file in [('train','ds102_train.csv'), ('dev','ds102_dev.csv'), ('test','ds102_test.csv')]},
        'fit_data': 'train only; no dev or test used to fit features, models, or calibration',
        'oof_raw_macro_f1': raw_score, 'oof_cross_calibration_macro_f1': cross_score,
        'calibration_bias': bias.tolist(), 'dev': dev_report,
        'dev_without_comment_overlap': metrics(dev_y[unique_dev], dev_pred[unique_dev]) if unique_dev.any() else None,
        'selection_caveat': 'CV estimates were used for model selection; dev was previously used to inspect baseline errors.',
        'submission_rows_verified': len(saved),
        'prediction_counts': {int(k): int(v) for k, v in saved.spam_label.value_counts().items()},
    }
    (output/'metrics_v2.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'Đã kiểm tra và lưu {len(saved)} dòng: {path}', flush=True)
    return report


## 5. Chạy

Giữ `DATA_DIR = None` trên Kaggle để tự tìm CSV dưới `/kaggle/input`. Nếu notebook gắn nhiều bộ dữ liệu cùng tên file, điền đường dẫn chính xác. Chạy CPU có thể mất vài phút, tùy máy. File cần nộp là **submission_v2.csv** trong Output.


In [ ]:
DATA_DIR = None
OUTPUT_DIR = "/kaggle/working" if Path("/kaggle/working").is_dir() else "output_v2"
args = ["--output-dir", OUTPUT_DIR, "--folds", "3"]
if DATA_DIR is not None:
    args += ["--data-dir", DATA_DIR]
result = main_v2(args)


In [ ]:
submission = pd.read_csv(Path(OUTPUT_DIR) / "submission_v2.csv")
display(submission.head())
print("Rows:", len(submission))
print("Dev F1-macro:", result["dev"]["macro_f1"])
